# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

In [1]:
"""
Question 1: How many reads map to HA, and how many reads map to NA?

This script analyzes a FASTQ file to count reads that correspond to 
influenza HA (hemagglutinin) and NA (neuraminidase) genes.

The analysis works by looking for the reverse-complement sequences of 
the gene-specific end sequences in each read.

References:
- BioPython documentation: https://biopython.org/
- Course lectures 8 and 9 on FASTQ file analysis
"""

from Bio import SeqIO
from Bio.Seq import reverse_complement

# Define the known end sequences for HA and NA genes
# These are the 3' end sequences of the original HA and NA genes
ha_end = "CCGGATTTGCATATAATGATGCACCAT"
na_end = "CACGATAGATAAATAATAGTGCACCAT"

# Since the reads come from the reverse end of the molecules, we need to look
# for the reverse complement of these sequences in our reads
ha_rc = reverse_complement(ha_end)
na_rc = reverse_complement(na_end)

print("Expected reverse complement sequences in reads:")
print(f"HA reverse complement: {ha_rc}")
print(f"NA reverse complement: {na_rc}")

# Initialize counters for HA and NA reads
ha_count = 0
na_count = 0

# Parse the FASTQ file and count reads matching each gene
fastq_file = "barcodes_R1.fastq"

for record in SeqIO.parse(fastq_file, "fastq"):
    read_seq = str(record.seq)
    
    # Check if this read contains the HA-specific sequence
    if ha_rc in read_seq:
        ha_count += 1
    # Check if this read contains the NA-specific sequence
    elif na_rc in read_seq:
        na_count += 1

# Display results
print("\n" + "="*50)
print("ANSWER TO QUESTION 1:")
print("="*50)
print(f"Number of reads mapping to HA: {ha_count}")
print(f"Number of reads mapping to NA: {na_count}")
print(f"Total mapped reads: {ha_count + na_count}")
print("="*50)


Expected reverse complement sequences in reads:
HA reverse complement: ATGGTGCATCATTATATGCAAATCCGG
NA reverse complement: ATGGTGCACTATTATTTATCTATCGTG

ANSWER TO QUESTION 1:
Number of reads mapping to HA: 5409
Number of reads mapping to NA: 4122
Total mapped reads: 9531


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [2]:
# your code here...
# Question 2: How many HA sequences did not have a valid barcode? 
# Also answer the same question for NA.

from Bio import SeqIO
from Bio.Seq import reverse_complement
import re

# Define the known end sequences for HA and NA genes
ha_end = "CCGGATTTGCATATAATGATGCACCAT"
na_end = "CACGATAGATAAATAATAGTGCACCAT"

# Reverse complements for identifying reads
ha_rc = reverse_complement(ha_end)
na_rc = reverse_complement(na_end)

# Spacer sequence that separates barcode from gene end
spacer = "GCGGCCGCCT"

# Counters for HA and NA
ha_total = 0
ha_valid_barcodes = 0
na_total = 0
na_valid_barcodes = 0

# Parse the FASTQ file
fastq_file = "barcodes_R1.fastq"

for record in SeqIO.parse(fastq_file, "fastq"):
    read_seq = str(record.seq)
    
    # Check if read matches HA
    if ha_rc in read_seq:
        ha_total += 1
        
        # Find where the spacer sequence begins
        spacer_pos = read_seq.find(spacer)
        if spacer_pos != -1:
            # The barcode is the sequence before the spacer
            barcode = read_seq[:spacer_pos]
            
            # Check if barcode is valid: exactly 16 nucleotides with only ATGCN
            if len(barcode) == 16 and re.match(r"^[ATGCN]{16}$", barcode):
                ha_valid_barcodes += 1
    
    # Check if read matches NA
    elif na_rc in read_seq:
        na_total += 1
        
        # Find where the spacer sequence begins
        spacer_pos = read_seq.find(spacer)
        if spacer_pos != -1:
            # The barcode is the sequence before the spacer
            barcode = read_seq[:spacer_pos]
            
            # Check if barcode is valid: exactly 16 nucleotides with only ATGCN
            if len(barcode) == 16 and re.match(r"^[ATGCN]{16}$", barcode):
                na_valid_barcodes += 1

# Calculate sequences without valid barcodes
ha_without_valid_barcode = ha_total - ha_valid_barcodes
na_without_valid_barcode = na_total - na_valid_barcodes

# Display results
print("HA sequences without valid barcode:", ha_without_valid_barcode)
print("NA sequences without valid barcode:", na_without_valid_barcode)

HA sequences without valid barcode: 163
NA sequences without valid barcode: 212


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [3]:
# Question 3: Which HA barcode has the most counts (and how many)? Same for NA.

from collections import defaultdict

ha_counts = defaultdict(int)
na_counts = defaultdict(int)

for record in SeqIO.parse(fastq_file, "fastq"):
    read_seq = str(record.seq)

    if ha_rc in read_seq:
        counts = ha_counts
    elif na_rc in read_seq:
        counts = na_counts
    else:
        continue

    spacer_pos = read_seq.find(spacer)
    if spacer_pos != -1:
        barcode = read_seq[:spacer_pos]
        if re.fullmatch(r"[ATGCN]{16}", barcode):
            counts[barcode] += 1

# Key with the maximum value
ha_top = max(ha_counts, key=ha_counts.get)
na_top = max(na_counts, key=na_counts.get)

print(f"HA top barcode: {ha_top} with {ha_counts[ha_top]} counts")
print(f"NA top barcode: {na_top} with {na_counts[na_top]} counts")

HA top barcode: TTAATGTCGGGTCGGG with 155 counts
NA top barcode: CCCGGGGAGAACTGGT with 152 counts
